<a href="https://colab.research.google.com/github/sayandxzzz/sayan/blob/main/AI_RECOMMENDATION_LOGIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")


movies['genres'] = movies['genres'].fillna('')


tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])


cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


def recommend_movies(movie_name, top_n=10):

    movie_index = movies[
        movies['title'].str.contains(movie_name, case=False, na=False)
    ].index

    if len(movie_index) == 0:
        print("Movie not found!")
        return

    idx = movie_index[0]

    cosine_sim = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    )

    similarity_scores = list(enumerate(cosine_sim[0]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    print("\nRecommended Movies:\n")

    for i, score in similarity_scores:

        movie_title = movies.iloc[i]['title']

        avg_rating = ratings[
            ratings['movieId'] == movies.iloc[i]['movieId']
        ]['rating'].mean()

        print(
            f"{movie_title} | Similarity: {round(score*100,2)}% | Rating: {round(avg_rating,2)}"
        )


movie = input("Enter a movie name: ")

recommend_movies(movie)

In [ ]:
movie_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()

movies = movies.merge(
    movie_ratings,
    on='movieId',
    how='left'
)

movies['rating'] = movies['rating'].fillna(0)

movies['features'] = (
    movies['genres'] +
    " " +
    movies['rating'].astype(str)
)